In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!pip install catboost -q

In [16]:
import pandas as pd
import numpy as np
from catboost import CatBoost, Pool
from sklearn.metrics import ndcg_score
import os

In [17]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"

df_train = pd.read_parquet(f"{DATA_DIR}/train_fe.parquet")
df_val   = pd.read_parquet(f"{DATA_DIR}/val_fe.parquet")
df_test  = pd.read_parquet(f"{DATA_DIR}/test_fe.parquet")
print(f"train: {df_train.shape}, val: {df_val.shape}, test: {df_test.shape}")

df_train = df_train.sort_values('srch_id').reset_index(drop=True)
df_val   = df_val.sort_values('srch_id').reset_index(drop=True)
df_test  = df_test.sort_values('srch_id').reset_index(drop=True)
exclude_cols = ['srch_id', 'prop_id', 'relevance', 'date_time',
                'click_bool', 'booking_bool', 'gross_bookings_usd', 'position']
features = [c for c in df_train.columns if c not in exclude_cols]

X_train, y_train = df_train[features].copy(), df_train['relevance']
X_val,   y_val   = df_val[features].copy(),   df_val['relevance']
X_test           = df_test[features].copy()

train: (4215569, 63), val: (742778, 63), test: (4959183, 58)


In [18]:
cat_features = ['site_id', 'prop_country_id', 'srch_destination_id']
cat_features = [c for c in cat_features if c in features]
for c in cat_features:
    for df in [X_train, X_val, X_test]:
        df[c] = df[c].fillna(-1).astype(int).astype(str)

print(f"Features: {len(features)}, categorical: {cat_features}")
print(f"position in features: {'position' in features}  (train/val use real, test filled with train mean)")
train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=df_train['srch_id'],
    cat_features=cat_features,
)
val_pool = Pool(
    data=X_val,
    label=y_val,
    group_id=df_val['srch_id'],
    cat_features=cat_features,
)
test_pool = Pool(
    data=X_test,
    cat_features=cat_features,
)

Features: 55, categorical: ['site_id', 'prop_country_id', 'srch_destination_id']
position in features: False  (train/val use real, test filled with train mean)


In [19]:
best_params = {'loss_function': 'YetiRank', 'eval_metric': 'NDCG:top=5;type=Exp', 'custom_metric': ['NDCG:top=5;type=Exp'], 'iterations': 5000, 'random_seed': 42, 'task_type': 'CPU', 'verbose': 100, 'use_best_model': True, 'od_type': 'Iter', 'od_wait': 200, 'depth': 10, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'random_strength': 1, 'task_type': 'GPU',}

In [20]:
model = CatBoost(best_params)
model.fit(train_pool, eval_set=val_pool)

print("Model trained")

Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.3029450	best: 0.3029450 (0)	total: 173ms	remaining: 14m 25s
100:	test: 0.3580970	best: 0.3580970 (100)	total: 14.5s	remaining: 11m 43s
200:	test: 0.3699098	best: 0.3699098 (200)	total: 28.8s	remaining: 11m 27s
300:	test: 0.3761226	best: 0.3761226 (300)	total: 43.1s	remaining: 11m 12s
400:	test: 0.3789536	best: 0.3790396 (397)	total: 57.5s	remaining: 10m 59s
500:	test: 0.3815931	best: 0.3815931 (500)	total: 1m 11s	remaining: 10m 45s
600:	test: 0.3836737	best: 0.3836737 (600)	total: 1m 26s	remaining: 10m 31s
700:	test: 0.3850988	best: 0.3851954 (695)	total: 1m 40s	remaining: 10m 17s
800:	test: 0.3862449	best: 0.3862981 (799)	total: 1m 55s	remaining: 10m 4s
900:	test: 0.3869629	best: 0.3871186 (897)	total: 2m 9s	remaining: 9m 50s
1000:	test: 0.3882462	best: 0.3882775 (990)	total: 2m 24s	remaining: 9m 36s
1100:	test: 0.3887082	best: 0.3888955 (1097)	total: 2m 39s	remaining: 9m 23s
1200:	test: 0.3897635	best: 0.3897903 (1195)	total: 2m 53s	remaining: 9m 9s
1300:	test: 0.3906404	b

In [21]:
df_val['predicted_score'] = model.predict(val_pool)

In [22]:
def calculate_representation_disparity(df, k=5):
    # prop_brand_bool is +1 for chains and 0 for independent, let's invert it
    df['is_independent'] = 1 - df['prop_brand_bool']

    # Calculate the base rate of independent hotels for each search
    base_rate = df.groupby('srch_id')['is_independent'].mean().rename('base_rate')
    # Sort the DataFrame by search ID and predicted score to get the top-k recommendations
    df_sorted = df.sort_values(by=['srch_id', 'predicted_score'], ascending=[True, False])
    # Get the top-k recommendations for each search
    top_k = df_sorted.groupby('srch_id').head(k)
    # Calculate the recommendation rate for independent hotels in the top-k recommendations
    rec_rate = top_k.groupby('srch_id')['is_independent'].mean().rename(f'rec_rate_at_{k}')
    # Combine the base rate and recommendation rate into a single DataFrame
    fairness_df = pd.merge(base_rate, rec_rate, on='srch_id', how='inner')
    # Calculate the representation disparity
    fairness_df['representation_disparity'] = fairness_df[f'rec_rate_at_{k}'] - fairness_df['base_rate']
    # calculate the final dataset-wide metric
    overall_disparity = fairness_df['representation_disparity'].mean()

    return overall_disparity, fairness_df

avg_disparity, fairness_df = calculate_representation_disparity(df_val, k=5)
print(f"Average Representation Disparity at k=5: {avg_disparity}")

print(fairness_df)

Average Representation Disparity at k=5: -0.024078074910170135
         base_rate  rec_rate_at_5  representation_disparity
srch_id                                                    
12        0.821429            0.6                 -0.221429
21        1.000000            1.0                  0.000000
28        0.117647            0.0                 -0.117647
40        0.812500            0.2                 -0.612500
47        0.903226            0.8                 -0.103226
...            ...            ...                       ...
332730    0.533333            0.6                  0.066667
332740    0.227273            0.0                 -0.227273
332755    0.125000            0.2                  0.075000
332763    1.000000            1.0                  0.000000
332781    0.133333            0.2                  0.066667

[29969 rows x 3 columns]


In [23]:
# ---------------------------------------------------------
# True Positive Rate Parity (Recall@5)
# ---------------------------------------------------------

# isolate the hotels that were actually booked
booked_hotels = df_val[df_val['relevance'] == 5].copy()

# get the top 5 predicted hotels for every search query
val_sorted = df_val.sort_values(by=['srch_id', 'predicted_score'], ascending=[True, False])
top_5 = val_sorted.groupby('srch_id').head(5)

top_5_list = top_5.groupby('srch_id')['prop_id'].apply(list).reset_index(name='top_5_props')

# merge the top 5 list back to the booked hotels to see if the booked hotel is in the top 5
tpr_df = pd.merge(booked_hotels, top_5_list, on='srch_id', how='inner')

tpr_df['is_in_top_5'] = tpr_df.apply(lambda row: row['prop_id'] in row['top_5_props'], axis=1)

# Calculate TPR for different groups based on prop_brand_bool (chain vs. independent)
tpr_bias = tpr_df.groupby('prop_brand_bool')['is_in_top_5'].mean()

print("--- True Positive Rate Parity (Recall@5) by Hotel Type ---")
print(tpr_bias.rename(index={0: 'Independent Hotels', 1: 'Chain Hotels'}))

--- True Positive Rate Parity (Recall@5) by Hotel Type ---
prop_brand_bool
Independent Hotels    0.607148
Chain Hotels          0.619459
Name: is_in_top_5, dtype: float64


In [24]:
# ---------------------------------------------------------
# Group Fairness (NDCG@5 Disparity)
# Focus: Chain vs. Independent (prop_brand_bool)
# ---------------------------------------------------------

# 1. Define a helper function to calculate NDCG@5 for a single search query
def calculate_search_ndcg(group):
    # ndcg_score expects 2D arrays: [true_scores] and [predicted_scores]
    y_true = np.asarray([group['relevance'].values])
    y_pred = np.asarray([group['predicted_score'].values])

    # Catch cases where a search only has 1 hotel (rare, but causes errors)
    if y_true.shape[1] < 2:
        return 1.0 if y_true[0][0] > 0 else 0.0

    return ndcg_score(y_true, y_pred, k=5)

# 2. Calculate NDCG@5 for all searches in the validation set
# (This might take a minute depending on the size of val_data)
ndcg_per_search = df_val.groupby('srch_id').apply(calculate_search_ndcg).reset_index(name='ndcg_5')

# 3. Merge with our 'booked_hotels' dataframe from Metric 1
# We do this so we can label each search by the TYPE of hotel the user actually booked
ndcg_fairness_df = pd.merge(ndcg_per_search, booked_hotels[['srch_id', 'prop_brand_bool']], on='srch_id', how='inner')

# 4. Calculate average NDCG@5 based on the booked hotel's brand
ndcg_bias = ndcg_fairness_df.groupby('prop_brand_bool')['ndcg_5'].mean()

print("\n--- Group Fairness (NDCG@5) ---")
print("Average ranking quality based on the type of hotel the user ultimately booked:")
print(ndcg_bias.rename(index={0: 'Booked an Independent (0)', 1: 'Booked a Chain (1)'}))


--- Group Fairness (NDCG@5) ---
Average ranking quality based on the type of hotel the user ultimately booked:
prop_brand_bool
Booked an Independent (0)    0.42730
Booked a Chain (1)           0.43391
Name: ndcg_5, dtype: float64


/tmp/ipykernel_843/1749029651.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ndcg_per_search = df_val.groupby('srch_id').apply(calculate_search_ndcg).reset_index(name='ndcg_5')


In [25]:
# ---------------------------------------------------------
# METRIC: Representation Disparity
# Focus: Promotion Bias (promotion_flag)
# ---------------------------------------------------------

def calculate_promotion_disparity(df, k=5):
    # Calculate the base rate of PROMOTED hotels (promotion_flag == 1) for each search
    base_rate = df.groupby('srch_id')['promotion_flag'].mean().rename('base_rate')

    # Sort the DataFrame by search ID and predicted score to get the top-k recommendations
    df_sorted = df.sort_values(by=['srch_id', 'predicted_score'], ascending=[True, False])

    # Get the top-k recommendations for each search
    top_k = df_sorted.groupby('srch_id').head(k)

    # Calculate the recommendation rate for promoted hotels in the top-k recommendations
    rec_rate = top_k.groupby('srch_id')['promotion_flag'].mean().rename(f'rec_rate_at_{k}')

    # Combine the base rate and recommendation rate into a single DataFrame
    fairness_df = pd.merge(base_rate, rec_rate, on='srch_id', how='inner')

    # Calculate the representation disparity
    # (Positive number means promoted hotels are OVER-represented in the Top 5)
    fairness_df['representation_disparity'] = fairness_df[f'rec_rate_at_{k}'] - fairness_df['base_rate']

    # Calculate the final dataset-wide metric
    overall_disparity = fairness_df['representation_disparity'].mean()

    return overall_disparity, fairness_df

# Run the function
avg_promo_disparity, promo_fairness_df = calculate_promotion_disparity(df_val, k=5)

print(f"Average Promotion Representation Disparity at k=5: {avg_promo_disparity:.4f}")
print("\n--- Breakdown by Search ---")
print(promo_fairness_df.head(10))

Average Promotion Representation Disparity at k=5: 0.1484

--- Breakdown by Search ---
         base_rate  rec_rate_at_5  representation_disparity
srch_id                                                    
12        0.464286            0.6                  0.135714
21        0.241379            0.6                  0.358621
28        0.352941            0.8                  0.447059
40        0.312500            0.6                  0.287500
47        0.258065            0.2                 -0.058065
56        0.166667            0.2                  0.033333
61        0.000000            0.0                  0.000000
63        0.461538            0.6                  0.138462
80        0.030303            0.0                 -0.030303
81        0.166667            0.0                 -0.166667


In [26]:
# ---------------------------------------------------------
# True Positive Rate Parity (Recall@5)
# ---------------------------------------------------------

# isolate the hotels that were actually booked
booked_hotels = df_val[df_val['relevance'] == 5].copy()

# get the top 5 predicted hotels for every search query
val_sorted = df_val.sort_values(by=['srch_id', 'predicted_score'], ascending=[True, False])
top_5 = val_sorted.groupby('srch_id').head(5)

top_5_list = top_5.groupby('srch_id')['prop_id'].apply(list).reset_index(name='top_5_props')

# merge the top 5 list back to the booked hotels to see if the booked hotel is in the top 5
tpr_df = pd.merge(booked_hotels, top_5_list, on='srch_id', how='inner')

tpr_df['is_in_top_5'] = tpr_df.apply(lambda row: row['prop_id'] in row['top_5_props'], axis=1)

# Calculate TPR for different groups based on promotion_flag
tpr_bias = tpr_df.groupby('promotion_flag')['is_in_top_5'].mean()

print("--- True Positive Rate Parity (Recall@5) by Promotion ---")
print(tpr_bias.rename(index={0: 'Not Promoted Hotels', 1: 'Promoted Hotels'}))

--- True Positive Rate Parity (Recall@5) by Promotion ---
promotion_flag
Not Promoted Hotels    0.569628
Promoted Hotels        0.720427
Name: is_in_top_5, dtype: float64


In [27]:
# ---------------------------------------------------------
# METRIC: Group Fairness (NDCG@5 Disparity)
# ---------------------------------------------------------

# We need the hotels that were ACTUALLY booked (relevance == 5)
booked_hotels = df_val[df_val['relevance'] == 5].copy()

ndcg_promo_df = pd.merge(ndcg_per_search, booked_hotels[['srch_id', 'promotion_flag']], on='srch_id', how='inner')

# Calculate average NDCG@5 based on whether the booked hotel was promoted or not
ndcg_promo_bias = ndcg_promo_df.groupby('promotion_flag')['ndcg_5'].mean()

print("--- Group Fairness (NDCG@5) by Promotion ---")
print("Average ranking quality based on whether the user ultimately booked a promoted hotel:")
# Remember: 0 is Not Promoted, 1 is Promoted
print(ndcg_promo_bias.rename(index={0: 'Booked Not Promoted (0)', 1: 'Booked Promoted (1)'}))

--- Group Fairness (NDCG@5) by Promotion ---
Average ranking quality based on whether the user ultimately booked a promoted hotel:
promotion_flag
Booked Not Promoted (0)    0.386209
Booked Promoted (1)        0.536280
Name: ndcg_5, dtype: float64


In [28]:
# ---------------------------------------------------------
# MITIGATION: Pre-processing (Sample Reweighting in CatBoost)
# Focus: Penalizing Revenue Bias
# ---------------------------------------------------------

# Calculate the weights for the training set
# Give a standard weight of 1.0 to promoted hotels.
# Give a higher weight (2.0) to NON-promoted hotels to force the model to learn organic patterns.
train_weights_promo = np.where(df_train['promotion_flag'] == 0, 2.0, 1.0)

# Create a NEW training pool that includes the weights!
train_pool_weighted = Pool(
    data=X_train,
    label=y_train,
    group_id=df_train['srch_id'],
    cat_features=cat_features,
    weight=train_weights_promo
)

print("Retraining CatBoost model with bias mitigation (Sample Weights)...")
mitigated_catboost = CatBoost(best_params)
mitigated_catboost.fit(train_pool_weighted, eval_set=val_pool)

df_val['mitigated_promo_score_preprocessing'] = mitigated_catboost.predict(val_pool)

Retraining CatBoost model with bias mitigation (Sample Weights)...


Pairwise losses don't support object weights.


Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because NDCG is/are not implemented for GPU
Metric NDCG:type=Base is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric NDCG:top=5;type=Exp is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.2824694	best: 0.2824694 (0)	total: 193ms	remaining: 16m 3s
100:	test: 0.3543687	best: 0.3545502 (99)	total: 14.7s	remaining: 11m 51s
200:	test: 0.3622299	best: 0.3623326 (199)	total: 29.2s	remaining: 11m 36s
300:	test: 0.3678761	best: 0.3678761 (300)	total: 43.7s	remaining: 11m 22s
400:	test: 0.3696669	best: 0.3698489 (396)	total: 58.3s	remaining: 11m 8s
500:	test: 0.3721610	best: 0.3721956 (498)	total: 1m 12s	remaining: 10m 54s
600:	test: 0.3740740	best: 0.3741820 (594)	total: 1m 27s	remaining: 10m 41s
700:	test: 0.3755173	best: 0.3756215 (697)	total: 1m 42s	remaining: 10m 27s
800:	test: 0.3764234	best: 0.3764936 (795)	total: 1m 57s	remaining: 10m 14s
900:	test: 0.3770301	best: 0.3770956 (899)	total: 2m 12s	remaining: 10m 1s
1000:	test: 0.3775937	best: 0.3777216 (988)	total: 2m 26s	remaining: 9m 47s
1100:	test: 0.3780888	best: 0.3783618 (1095)	total: 2m 41s	remaining: 9m 33s
1200:	test: 0.3790298	best: 0.3791699 (1196)	total: 2m 56s	remaining: 9m 18s
1300:	test: 0.3799106	b

In [29]:
# ---------------------------------------------------------
# 1. MITIGATED REPRESENTATION DISPARITY
# ---------------------------------------------------------
def calculate_mitigated_promotion_disparity(df, k=5):
    base_rate = df.groupby('srch_id')['promotion_flag'].mean().rename('base_rate')

    # SORT USING THE NEW SCORE
    df_sorted = df.sort_values(by=['srch_id', 'mitigated_promo_score_preprocessing'], ascending=[True, False])
    top_k = df_sorted.groupby('srch_id').head(k)
    rec_rate = top_k.groupby('srch_id')['promotion_flag'].mean().rename(f'rec_rate_at_{k}')

    fairness_df = pd.merge(base_rate, rec_rate, on='srch_id', how='inner')
    fairness_df['representation_disparity'] = fairness_df[f'rec_rate_at_{k}'] - fairness_df['base_rate']
    return fairness_df['representation_disparity'].mean()

mitigated_promo_disparity = calculate_mitigated_promotion_disparity(df_val, k=5)
print(f"1. Mitigated Representation Disparity: {mitigated_promo_disparity:.4f}")
print("   (Goal: Get closer to 0.0000. Originally: 0.0987)\n")


# ---------------------------------------------------------
# 2. MITIGATED TRUE POSITIVE RATE PARITY (Recall@5)
# ---------------------------------------------------------
booked_hotels = df_val[df_val['relevance'] == 5].copy()

# SORT USING THE NEW SCORE
val_sorted_mitigated = df_val.sort_values(by=['srch_id', 'mitigated_promo_score_preprocessing'], ascending=[True, False])
top_5_mitigated = val_sorted_mitigated.groupby('srch_id').head(5)

top_5_list_mitigated = top_5_mitigated.groupby('srch_id')['prop_id'].apply(list).reset_index(name='top_5_props')
tpr_df_mitigated = pd.merge(booked_hotels, top_5_list_mitigated, on='srch_id', how='inner')
tpr_df_mitigated['is_in_top_5'] = tpr_df_mitigated.apply(lambda row: row['prop_id'] in row['top_5_props'], axis=1)

tpr_bias_mitigated = tpr_df_mitigated.groupby('promotion_flag')['is_in_top_5'].mean()
print("2. Mitigated True Positive Rate Parity (Recall@5):")
print(tpr_bias_mitigated.rename(index={0: 'Not Promoted Hotels', 1: 'Promoted Hotels'}))
print("   (Goal: Close the gap between these two numbers, before: Not Promoted Hotels    0.551139 Promoted Hotels        0.673300)\n")


# ---------------------------------------------------------
# 3. MITIGATED GROUP FAIRNESS (NDCG@5)
# ---------------------------------------------------------
def calculate_mitigated_search_ndcg(group):
    y_true = np.asarray([group['relevance'].values])
    # USE THE NEW SCORE FOR PREDICTIONS
    y_pred = np.asarray([group['mitigated_promo_score_preprocessing'].values])

    if y_true.shape[1] < 2:
        return 1.0 if y_true[0][0] > 0 else 0.0
    return ndcg_score(y_true, y_pred, k=5)

# Calculate new NDCG per search
mitigated_ndcg_per_search = df_val.groupby('srch_id').apply(calculate_mitigated_search_ndcg, include_groups=False).reset_index(name='ndcg_5')

# Merge with booked hotels to split by promotion status
mitigated_ndcg_promo_df = pd.merge(mitigated_ndcg_per_search, booked_hotels[['srch_id', 'promotion_flag']], on='srch_id', how='inner')
mitigated_ndcg_promo_bias = mitigated_ndcg_promo_df.groupby('promotion_flag')['ndcg_5'].mean()

print("3. Mitigated Group Fairness (NDCG@5):")
print(mitigated_ndcg_promo_bias.rename(index={0: 'Booked Not Promoted (0)', 1: 'Booked Promoted (1)'}))
print("   (Goal: Increase the score for the 'Not Promoted' group to close the gap) before: Booked Not Promoted (0)    0.374786 Booked Promoted (1)        0.491344")

1. Mitigated Representation Disparity: 0.0066
   (Goal: Get closer to 0.0000. Originally: 0.0987)

2. Mitigated True Positive Rate Parity (Recall@5):
promotion_flag
Not Promoted Hotels    0.615955
Promoted Hotels        0.558987
Name: is_in_top_5, dtype: float64
   (Goal: Close the gap between these two numbers, before: Not Promoted Hotels    0.551139 Promoted Hotels        0.673300)

3. Mitigated Group Fairness (NDCG@5):
promotion_flag
Booked Not Promoted (0)    0.430116
Booked Promoted (1)        0.383752
Name: ndcg_5, dtype: float64
   (Goal: Increase the score for the 'Not Promoted' group to close the gap) before: Booked Not Promoted (0)    0.374786 Booked Promoted (1)        0.491344
